# 자율주행컴퓨팅 실습 1 — Two-stage vs One-stage Object Detection (Faster R-CNN · RetinaNet)

같은 이미지 한 장(trace image)을 두 검출기에 넣고, 각 단계에서 만들어지는 텐서를 직접 꺼내 비교합니다.

**Part 1. Two-stage — Faster R-CNN** (1 ~ 9절)

| 단계 | 질문 | 섹션 |
|---|---|---|
| Backbone + FPN | 이미지가 어떤 크기의 feature map 들로 바뀌는가? | 2 |
| Anchor | 학습 전부터 존재하는 "후보 박스 템플릿"은 어떻게 깔리는가? | 3 |
| RPN target | 어떤 anchor 가 positive/negative 가 되고, regression 정답은 무엇인가? | 4 |
| Proposal | RPN 이 수십만 개 anchor 중 무엇을 2단계로 넘기는가? | 5 |
| 2nd stage | 넘겨받은 proposal 을 어떻게 다시 분류·보정하는가? | 6 |
| 학습 전 vs 후 | 같은 입력에 대해 무엇이 달라지는가? | 7, 8 |

**Part 2. One-stage — RetinaNet** (10 ~ 17절)

| 단계 | 질문 | 섹션 |
|---|---|---|
| 구조 | Part 1 의 어느 모듈이 사라졌는가? | 10 |
| Dense prediction | 한 위치(한 anchor)는 무엇을 출력하는가? | 12 |
| Target & Focal loss | 샘플링 없이 모든 anchor 를 쓰면 어떤 문제가 생기고, 어떻게 해결하는가? | 13 |
| 후처리 | 수만 개의 dense 출력이 어떻게 최종 검출이 되는가? | 14 |
| 학습 전 vs 후 | 같은 입력에 대해 무엇이 달라지는가? | 15, 16 |

> ⚙️ Colab 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 를 선택한 뒤, **위에서부터 순서대로** 실행하세요. (Part 2 는 Part 1 에서 만든 데이터셋과 함수를 그대로 사용합니다.)

## 0. 환경 설정

In [ ]:
import os, math, random, copy, time, glob, zipfile, urllib.request
import numpy as np
import torch, torchvision
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import torchvision.transforms.functional as TF
from torchvision.models import ResNet50_Weights
from torchvision.models.detection import (fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights,
                                          retinanet_resnet50_fpn, RetinaNet_ResNet50_FPN_Weights)
from torchvision.models.detection.rpn import concat_box_prediction_layers
from torchvision.ops import box_iou, clip_boxes_to_image, sigmoid_focal_loss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 0
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_all()
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| device:", device)
if device.type == "cpu":
    print("⚠️ GPU 가 잡히지 않았습니다. 런타임 유형을 T4 GPU 로 바꿔주세요 (CPU 로는 학습이 매우 느립니다).")

### 실습 설정
- 데이터셋: **Penn-Fudan 보행자 데이터셋** (170장, 아래 셀에서 자동 다운로드)
- `EPOCHS` : 8절(Faster R-CNN)과 16절(RetinaNet)에서 각각 직접 학습시킬 epoch 수. `SNAP_EPOCHS` 시점의 가중치를 저장해 두고 학습 전/중/후를 비교합니다.

In [ ]:
DATA_ROOT = "/content/data"

EPOCHS      = 15
SNAP_EPOCHS = [0, 1, 3, 8, EPOCHS]   # 이 epoch 들의 가중치를 저장해 비교 (0 = 학습 전)
BATCH_SIZE  = 2
LR_FRCNN    = 0.01                   # Part 1 Faster R-CNN 학습률
LR_RETINA   = 0.005                  # Part 2 RetinaNet 학습률
NUM_VAL     = 30                     # 검증용 이미지 수
MIN_SIZE, MAX_SIZE = 600, 1000       # 모델 입력 크기 (짧은 변 600 px)

# Part 1. Two-stage detector 해부하기 — Faster R-CNN

## 1. 데이터셋과 모델 준비

In [ ]:
class PennFudan(torch.utils.data.Dataset):
    CLASSES = ["__background__", "pedestrian"]
    def __init__(self, root, names, train=False):
        self.root, self.names, self.train = root, names, train
    def __len__(self): return len(self.names)
    def __getitem__(self, i):
        n = self.names[i]
        img = Image.open(os.path.join(self.root, "PNGImages", n + ".png")).convert("RGB")
        mask = np.array(Image.open(os.path.join(self.root, "PedMasks", n + "_mask.png")))
        boxes = []
        for k in np.unique(mask)[1:]:                    # 0 = 배경, 1..K = 사람 instance
            ys, xs = np.where(mask == k)
            boxes.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1])
        boxes = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels = torch.ones(len(boxes), dtype=torch.int64)
        return _maybe_flip(TF.to_tensor(img), boxes, labels, self.train)

def _maybe_flip(img, boxes, labels, train):
    if train and random.random() < 0.5:                    # 학습 시 좌우 반전 augmentation
        img = img.flip(-1); W = img.shape[-1]
        boxes = boxes.clone(); boxes[:, [0, 2]] = W - boxes[:, [2, 0]]
    return img, {"boxes": boxes, "labels": labels}

root = os.path.join(DATA_ROOT, "PennFudanPed")
if not os.path.isdir(root):
    os.makedirs(DATA_ROOT, exist_ok=True)
    zp = os.path.join(DATA_ROOT, "PennFudanPed.zip")
    urllib.request.urlretrieve("https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip", zp)
    zipfile.ZipFile(zp).extractall(DATA_ROOT)
names = sorted(f[:-4] for f in os.listdir(os.path.join(root, "PNGImages")))
random.Random(SEED).shuffle(names)
val_ds   = PennFudan(root, names[:NUM_VAL], train=False)
train_ds = PennFudan(root, names[NUM_VAL:], train=True)
CLASSES = PennFudan.CLASSES
print(f"Penn-Fudan: train {len(train_ds)} / val {len(val_ds)} images, classes = {CLASSES[1:]}")

In [ ]:
# ---- 시각화 helper ----
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
LEVEL_NAMES = ["P2", "P3", "P4", "P5", "P6"]
LEVEL_COLORS = ["tab:red", "tab:orange", "tab:green", "tab:blue", "tab:purple"]
GT_COLORS = plt.cm.tab10.colors

def to_display(image_list, i=0):
    # model.transform 을 거친(정규화+리사이즈+패딩) 텐서를 다시 보기 좋은 이미지로
    h, w = image_list.image_sizes[i]
    t = image_list.tensors[i, :, :h, :w].detach().cpu()
    return (t * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

def imshow(ax, img, title=None):
    ax.imshow(img); ax.axis("off")
    if title: ax.set_title(title, fontsize=10)

def draw_boxes(ax, boxes, color="lime", lw=1.5, ls="-", alpha=1.0, texts=None, fs=7):
    boxes = boxes.detach().cpu() if torch.is_tensor(boxes) else boxes
    if texts is not None and len(boxes) > 20: texts = None        # 박스가 너무 많으면 글자는 생략
    for k, b in enumerate(boxes):
        x1, y1, x2, y2 = [float(v) for v in b]
        c = color[k] if isinstance(color, (list, tuple)) and not isinstance(color[0], float) else color
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                       edgecolor=c, linewidth=lw, linestyle=ls, alpha=alpha))
        if texts is not None:
            ax.text(x1, y1 - 2, texts[k], color="white", fontsize=fs,
                    bbox=dict(facecolor=c, alpha=0.75, pad=1, edgecolor="none"))

def to_dev(tgt):
    return {k: v.to(device) for k, v in tgt.items()}

### 모델 2개 만들기
`fasterrcnn_resnet50_fpn(weights=None, weights_backbone=IMAGENET)` 은 **ResNet-50 body 만 ImageNet 가중치**를 쓰고,
FPN · RPN · RoI head 는 전부 랜덤 초기화합니다. 강의에서 말한 "pretrained backbone 위에 RPN 을 처음 붙인 상태"가 바로 이것입니다.

In [ ]:
def build_init_model():
    seed_all()
    return fasterrcnn_resnet50_fpn(weights=None, weights_backbone=ResNet50_Weights.IMAGENET1K_V1,
                                   num_classes=len(CLASSES), min_size=MIN_SIZE, max_size=MAX_SIZE).to(device)

model_init = build_init_model().eval()
INIT_STATE = {k: v.detach().cpu().clone() for k, v in model_init.state_dict().items()}   # epoch 0 스냅샷

COCO_W = FasterRCNN_ResNet50_FPN_Weights.COCO_V1
model_coco = fasterrcnn_resnet50_fpn(weights=COCO_W, min_size=MIN_SIZE, max_size=MAX_SIZE).to(device).eval()
COCO_CLASSES = COCO_W.meta["categories"]

def n_params(m): return sum(p.numel() for p in m.parameters()) / 1e6
print(f"backbone.body (ResNet-50) : {n_params(model_init.backbone.body):6.2f} M  ← ImageNet pretrained")
print(f"backbone.fpn              : {n_params(model_init.backbone.fpn):6.2f} M  ← random init")
print(f"rpn                       : {n_params(model_init.rpn):6.2f} M  ← random init")
print(f"roi_heads                 : {n_params(model_init.roi_heads):6.2f} M  ← random init")

## 2. Trace image 와 Backbone + FPN feature

Faster R-CNN 의 `forward` 는 내부적으로 다음 순서로 동작합니다. 이 노트북은 이 줄들을 **하나씩 따로 호출**합니다.
```python
images, targets = model.transform(images, targets)      # resize + normalize + batch padding
features        = model.backbone(images.tensors)        # ResNet-50 + FPN → {P2..P6}
proposals, l1   = model.rpn(images, features, targets)   # 1st stage
detections, l2  = model.roi_heads(features, proposals, images.image_sizes, targets)  # 2nd stage
```

In [ ]:
# GT 가 2~4개 있는 val 이미지를 trace image 로 선택
TRACE_IDX = next(i for i in range(len(val_ds)) if 2 <= len(val_ds[i][1]["boxes"]) <= 4)
img, tgt = val_ds[TRACE_IDX]

with torch.no_grad():
    image_list, targets_t = model_init.transform([img.to(device)], [to_dev(tgt)])
    feats_init = model_init.backbone(image_list.tensors)
    feats_coco = model_coco.backbone(image_list.tensors)

gt_boxes = targets_t[0]["boxes"]
disp = to_display(image_list)
print("원본 이미지:", tuple(img.shape[1:]), "→ transform 후:", tuple(image_list.image_sizes[0]),
      "→ 패딩된 batch tensor:", tuple(image_list.tensors.shape[-2:]))
for k, (name, f) in enumerate(zip(LEVEL_NAMES, feats_coco.values())):
    stride = image_list.tensors.shape[-1] / f.shape[-1]
    print(f"  {name}: {tuple(f.shape)}  stride ≈ {stride:.0f}")

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
imshow(ax, disp, f"Trace image (val #{TRACE_IDX}) with GT")
draw_boxes(ax, gt_boxes, color=[GT_COLORS[i] for i in range(len(gt_boxes))], lw=2.5,
           texts=[f"GT{i}" for i in range(len(gt_boxes))])
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6.5))
for r, (tag, feats) in enumerate([("ImageNet body + random FPN", feats_init), ("COCO-trained", feats_coco)]):
    for c, (name, f) in enumerate(zip(LEVEL_NAMES, feats.values())):
        a = f[0].abs().mean(0).cpu()
        axes[r, c].imshow(a, cmap="viridis"); axes[r, c].axis("off")
        axes[r, c].set_title(f"{tag}\n{name} {tuple(a.shape)}", fontsize=9)
plt.suptitle("Mean |activation| per FPN level", y=1.02); plt.tight_layout(); plt.show()

> 🤔 **생각해보기** — P2 는 칸이 촘촘하고(해상도 높음) P6 는 칸이 큽니다(해상도 낮음). 작은 보행자와 큰 보행자는 각각 어느 레벨에서 찾는 것이 유리할까요?

## 3. Anchor 템플릿은 어떻게 만들어지는가

Anchor 는 **학습 파라미터가 아닙니다.** 모델을 만드는 순간 정해지는 고정된 박스 템플릿입니다.
- 각 FPN 레벨마다 크기 1개 × 비율 3개(0.5, 1, 2) = 위치당 **A = 3개**의 template
- 이 template 을 feature map 의 **모든 위치**(stride 간격)에 복사 → 이미지 한 장에 수십만 개 anchor

In [ ]:
ag = model_init.rpn.anchor_generator
print("sizes         :", ag.sizes)
print("aspect_ratios :", ag.aspect_ratios[0])

feature_list = list(feats_init.values())
anchors = ag(image_list, feature_list)[0]                  # (전체 anchor 수, 4) - x1,y1,x2,y2
A = ag.num_anchors_per_location()
grid_sizes = [tuple(f.shape[-2:]) for f in feature_list]
num_per_level = [h * w * a for (h, w), a in zip(grid_sizes, A)]
offsets = np.cumsum([0] + num_per_level)
for name, gs, a, n in zip(LEVEL_NAMES, grid_sizes, A, num_per_level):
    print(f"  {name}: grid {gs[0]:3d}x{gs[1]:3d} x {a} anchors = {n:7,d}")
print(f"  총 anchor 수 = {len(anchors):,}")

# (a) template 15개를 원점 기준으로 그리기
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ax = axes[0]
for l, ca in enumerate(ag.cell_anchors):
    draw_boxes(ax, ca.cpu(), color=LEVEL_COLORS[l], lw=1.8)
    ax.plot([], [], color=LEVEL_COLORS[l], label=f"{LEVEL_NAMES[l]}: size {ag.sizes[l][0]}")
ax.set_xlim(-400, 400); ax.set_ylim(400, -400); ax.set_aspect("equal"); ax.grid(alpha=.3)
ax.legend(fontsize=9); ax.set_title("cell_anchors: 5 levels x 3 ratios (centered at 0)")

# (b) 가장 큰 GT 의 중심 위치에 각 레벨의 anchor 3개씩 = 15개를 올려보기
g = int((gt_boxes[:, 2:] - gt_boxes[:, :2]).prod(1).argmax())
cx, cy = ((gt_boxes[g, :2] + gt_boxes[g, 2:]) / 2).tolist()
imshow(axes[1], disp, f"15 anchors at the center of GT{g}")
draw_boxes(axes[1], gt_boxes[g:g+1], color="white", lw=3)
for l, (H, W) in enumerate(grid_sizes):
    s = image_list.tensors.shape[-1] / W
    x, y = min(int(round(cx / s)), W - 1), min(int(round(cy / s)), H - 1)
    idx = offsets[l] + (y * W + x) * A[l] + np.arange(A[l])
    ious = box_iou(anchors[idx], gt_boxes[g:g+1])[:, 0]
    draw_boxes(axes[1], anchors[idx], color=LEVEL_COLORS[l], lw=1.5)
    print(f"{LEVEL_NAMES[l]} anchors vs GT{g} IoU:", [f"{v:.2f}" for v in ious.tolist()])
axes[1].set_xlim(0, disp.shape[1]); axes[1].set_ylim(disp.shape[0], 0)
plt.tight_layout(); plt.show()

In [ ]:
# (c) 레벨별 anchor 중심 grid 밀도
centers = (anchors[:, :2] + anchors[:, 2:]) / 2
fig, axes = plt.subplots(1, 5, figsize=(20, 4.5))
for l in range(5):
    c = centers[offsets[l]:offsets[l + 1]:A[l]].cpu()        # 위치당 1개만 (A개는 중심이 같음)
    imshow(axes[l], disp, f"{LEVEL_NAMES[l]}: {len(c):,} locations")
    axes[l].scatter(c[:, 0], c[:, 1], s=0.5 if l == 0 else 3, color=LEVEL_COLORS[l])
plt.tight_layout(); plt.show()

> 🤔 **생각해보기** — 위 출력에서 GT 와 IoU 가 가장 높은 anchor 는 어느 레벨, 어느 비율(0.5 / 1 / 2)이었나요?
> (torchvision 의 비율은 **높이/너비** 입니다. 세로로 긴 보행자에게는 어떤 비율이 잘 맞을까요?)

## 4. RPN 의 학습 정답(target)은 어떻게 만들어지는가

**① Label 할당** — 각 anchor 와 모든 GT 의 IoU 를 계산해서
- max IoU ≥ 0.7 → **positive (1)**
- max IoU < 0.3 → **negative (0)**
- 그 사이 → **ignore (-1)**, loss 계산에서 제외
- 단, 어떤 GT 와도 0.7 을 넘는 anchor 가 없으면, 그 GT 와 IoU 가 가장 높은 anchor 를 강제로 positive 로 지정 (`allow_low_quality_matches`)

**② Sampling** — positive 는 매우 적고 negative 는 압도적으로 많으므로, 이미지당 256개를 positive 최대 50% 로 샘플링

**③ Regression target** — positive anchor $a$ 와 매칭된 GT $g$ 사이의 상대 오프셋:
$$t_x = \frac{g_x - a_x}{a_w},\quad t_y = \frac{g_y - a_y}{a_h},\quad t_w = \log\frac{g_w}{a_w},\quad t_h = \log\frac{g_h}{a_h}$$
(중심 좌표 기준. 이 target 은 **모델 가중치와 무관**합니다 — anchor 와 GT 만으로 결정됩니다.)

In [ ]:
rpn = model_init.rpn
print(f"fg IoU ≥ {rpn.proposal_matcher.high_threshold}, bg IoU < {rpn.proposal_matcher.low_threshold}, "
      f"batch {rpn.fg_bg_sampler.batch_size_per_image}, pos fraction {rpn.fg_bg_sampler.positive_fraction}")

labels, matched_gt = rpn.assign_targets_to_anchors([anchors], targets_t)
labels, matched_gt = labels[0], matched_gt[0]
iou = box_iou(gt_boxes, anchors)                 # (G, N)
max_iou, gt_idx = iou.max(0)

pos, neg, ign = labels == 1, labels == 0, labels == -1
print(f"\npositive {int(pos.sum()):,} | negative {int(neg.sum()):,} | ignore {int(ign.sum()):,}  (총 {len(labels):,})")
print(f"  → 그중 IoU<0.7 인데 low-quality match 로 positive 가 된 anchor: {int((pos & (max_iou < 0.7)).sum())}")
print("  레벨별 positive 수:", {LEVEL_NAMES[l]: int(pos[offsets[l]:offsets[l+1]].sum()) for l in range(5)})

def rand_subset(mask, k):
    idx = torch.where(mask)[0]
    return idx[torch.randperm(len(idx), device=idx.device)[:k]]

seed_all()
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
imshow(axes[0], disp, f"Positive anchors ({int(pos.sum())}) colored by matched GT")
pi = torch.where(pos)[0]
draw_boxes(axes[0], anchors[pi], color=[GT_COLORS[int(gt_idx[i])] for i in pi], lw=1)
draw_boxes(axes[0], gt_boxes, color="white", lw=2.5)
imshow(axes[1], disp, f"Ignored anchors (random 150 of {int(ign.sum()):,})")
draw_boxes(axes[1], anchors[rand_subset(ign, 150)], color="yellow", lw=0.7)
draw_boxes(axes[1], gt_boxes, color="white", lw=2.5)
imshow(axes[2], disp, f"Negative anchors (random 150 of {int(neg.sum()):,})")
draw_boxes(axes[2], anchors[rand_subset(neg, 150)], color="red", lw=0.7)
draw_boxes(axes[2], gt_boxes, color="white", lw=2.5)
for ax in axes: ax.set_xlim(0, disp.shape[1]); ax.set_ylim(disp.shape[0], 0)
plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 3))
plt.hist(max_iou.cpu().numpy(), bins=50, log=True, color="gray")
plt.axvline(0.3, color="red", ls="--", label="bg < 0.3"); plt.axvline(0.7, color="green", ls="--", label="fg ≥ 0.7")
plt.xlabel("max IoU with any GT"); plt.ylabel("# anchors (log)"); plt.legend(); plt.title("Anchor IoU distribution")
plt.show()

In [ ]:
# ② 256개 샘플링 — 실제로 loss 에 쓰이는 anchor
seed_all()
pos_m, neg_m = rpn.fg_bg_sampler([labels])
pos_m, neg_m = pos_m[0].bool(), neg_m[0].bool()
print(f"샘플링된 positive {int(pos_m.sum())}, negative {int(neg_m.sum())}  → 합계 {int(pos_m.sum() + neg_m.sum())}")

fig, ax = plt.subplots(1, 1, figsize=(7, 7))
imshow(ax, disp, "Anchors actually used in RPN loss (green=pos, red=neg)")
draw_boxes(ax, anchors[neg_m], color="red", lw=0.5, alpha=0.6)
draw_boxes(ax, anchors[pos_m], color="lime", lw=1)
ax.set_xlim(0, disp.shape[1]); ax.set_ylim(disp.shape[0], 0); plt.show()

In [ ]:
# ③ Regression target
reg_t = rpn.box_coder.encode([matched_gt], [anchors])[0]       # (N, 4) — positive 에서만 의미 있음
print("RPN box_coder weights:", rpn.box_coder.weights)

def cxcywh(b): return torch.stack([(b[..., 0] + b[..., 2]) / 2, (b[..., 1] + b[..., 3]) / 2,
                                    b[..., 2] - b[..., 0], b[..., 3] - b[..., 1]], -1)

# IoU 가 높은 / 중간 / 낮은 positive anchor 를 하나씩 골라 비교
pi_sorted = pi[max_iou[pi].argsort(descending=True)]
examples = [pi_sorted[0], pi_sorted[len(pi_sorted) // 2], pi_sorted[-1]]
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, i in zip(axes, examples):
    a, gbox, t = anchors[i], matched_gt[i], reg_t[i]
    ac, gc = cxcywh(a), cxcywh(gbox)
    manual = torch.stack([(gc[0]-ac[0])/ac[2], (gc[1]-ac[1])/ac[3], torch.log(gc[2]/ac[2]), torch.log(gc[3]/ac[3])])
    decoded = rpn.box_coder.decode_single(t[None], a[None])[0]
    print(f"anchor #{int(i)} (IoU {max_iou[i]:.2f})  target (tx,ty,tw,th) = {np.round(t.tolist(), 3)}"
          f" | 수식 직접 계산 = {np.round(manual.tolist(), 3)} | decode → GT 복원 오차 {float((decoded-gbox).abs().max()):.1e}")
    imshow(ax, disp, f"IoU {max_iou[i]:.2f}\n t=({t[0]:.2f}, {t[1]:.2f}, {t[2]:.2f}, {t[3]:.2f})")
    draw_boxes(ax, a[None], color="cyan", lw=2, ls="--")
    draw_boxes(ax, gbox[None], color="white", lw=2.5)
    ax.annotate("", xy=gc[:2].tolist(), xytext=ac[:2].tolist(),
                arrowprops=dict(arrowstyle="->", color="yellow", lw=2.5))
    pad = 60
    ax.set_xlim(float(min(a[0], gbox[0])) - pad, float(max(a[2], gbox[2])) + pad)
    ax.set_ylim(float(max(a[3], gbox[3])) + pad, float(min(a[1], gbox[1])) - pad)
plt.suptitle("anchor (cyan dashed) → GT (white): this offset is the RPN regression target", y=1.0)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(16, 2.8))
for k, n in enumerate(["tx", "ty", "tw", "th"]):
    axes[k].hist(reg_t[pos, k].cpu().numpy(), bins=30, color="tab:blue"); axes[k].set_title(f"{n} (positives)")
plt.tight_layout(); plt.show()

> 🤔 **생각해보기** — 이 이미지에서 positive anchor 와 negative anchor 는 각각 몇 개였나요?
> negative 가 훨씬 많은데 전부 loss 에 넣으면, 모델은 "전부 배경" 이라고 답하는 쪽으로 치우치지 않을까요? 그래서 256개만 골라 씁니다.

## 5. RPN 의 예측 → Proposal 선별 (학습 전 vs 학습 후)

RPN head 는 각 anchor 마다 **objectness 점수 1개 + 오프셋 4개**를 예측합니다.
Proposal 은 다음 과정으로 만들어집니다.
1. 예측 오프셋으로 anchor 를 이동시켜 박스로 decode
2. 레벨별 objectness 상위 k개 선택 (`pre_nms_top_n`)
3. NMS (IoU 0.7) 로 중복 제거
4. 최종 상위 `post_nms_top_n` 개를 2nd stage 로 전달 (train 2000 / test 1000)

In [ ]:
@torch.no_grad()
def rpn_dissect(model, image_list, targets=None):
    model.eval()
    feats = model.backbone(image_list.tensors)
    fl = list(feats.values())
    obj, deltas = model.rpn.head(fl)                                   # 레벨별 (1,A,H,W), (1,4A,H,W)
    anc = model.rpn.anchor_generator(image_list, fl)
    n_per = [o[0].numel() for o in obj]
    obj_flat, deltas_flat = concat_box_prediction_layers(obj, deltas)  # anchor 순서와 동일하게 펼침
    props_all = model.rpn.box_coder.decode(deltas_flat, anc).view(len(anc), -1, 4)
    boxes, scores = model.rpn.filter_proposals(props_all, obj_flat, image_list.image_sizes, n_per)
    out = dict(feats=feats, obj=obj, obj_scores=torch.sigmoid(obj_flat.view(-1)),
               props_all=clip_boxes_to_image(props_all[0], image_list.image_sizes[0]),
               proposals=boxes[0], proposal_scores=scores[0])
    if targets is not None:                                            # 같은 이미지에서의 RPN loss
        torch.manual_seed(SEED); model.rpn.train()
        _, losses = model.rpn(image_list, feats, targets); model.rpn.eval()
        out["losses"] = {k: round(v.item(), 4) for k, v in losses.items()}
    return out

R = {"init": rpn_dissect(model_init, image_list, targets_t), "coco": rpn_dissect(model_coco, image_list, targets_t)}
print(f"pre_nms_top_n(test)={model_init.rpn._pre_nms_top_n['testing']}, "
      f"post_nms_top_n(test)={model_init.rpn._post_nms_top_n['testing']}, NMS IoU={model_init.rpn.nms_thresh}")
for k in R: print(f"[{k}] RPN loss on trace image: {R[k]['losses']} | #proposals: {len(R[k]['proposals'])}")

In [ ]:
# objectness heatmap — 각 위치에서 A개 anchor 중 최대 objectness
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for r, k in enumerate(R):
    for l in range(5):
        hm = torch.sigmoid(R[k]["obj"][l][0]).max(0).values.cpu()
        H, W = hm.shape; s = image_list.tensors.shape[-1] / W
        ax = axes[r, l]; imshow(ax, disp)
        im = ax.imshow(hm, cmap="jet", alpha=0.5, vmin=0, vmax=1, extent=(0, W * s, H * s, 0))
        ax.set_xlim(0, disp.shape[1]); ax.set_ylim(disp.shape[0], 0)
        ax.set_title(f"[{k}] {LEVEL_NAMES[l]} objectness (max {hm.max():.2f})", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# (a) 학습 전 vs 학습 후 — 2nd stage 로 넘어가는 proposal (NMS 후 상위 50개)
TOPK = 50
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, k in zip(axes, R):
    imshow(ax, disp, f"[{k}] top-{TOPK} proposals sent to the 2nd stage")
    draw_boxes(ax, R[k]["proposals"][:TOPK], color="cyan", lw=1)
    draw_boxes(ax, gt_boxes, color="white", lw=2.5, ls="--")
plt.tight_layout(); plt.show()

# (b) NMS 의 효과 — 학습된 모델(coco)에서 "사람 한 명에게 겹친 박스" 만 모아 NMS 전/후 비교
Rc, SC_THR = R["coco"], 0.5
def boxes_on_gt(boxes, scores, gi):
    return (box_iou(boxes, gt_boxes[gi:gi + 1])[:, 0] > 0.5) & (scores > SC_THR)
n_before = [int(boxes_on_gt(Rc["props_all"], Rc["obj_scores"], gi).sum()) for gi in range(len(gt_boxes))]
n_after  = [int(boxes_on_gt(Rc["proposals"], Rc["proposal_scores"], gi).sum()) for gi in range(len(gt_boxes))]
for gi in range(len(gt_boxes)): print(f"GT{gi}: NMS 전 {n_before[gi]:3d}개 → NMS 후 {n_after[gi]:3d}개")
g = int(np.argmax(n_before))
bm = boxes_on_gt(Rc["props_all"], Rc["obj_scores"], g); am = boxes_on_gt(Rc["proposals"], Rc["proposal_scores"], g)

fig, axes = plt.subplots(1, 3, figsize=(21, 7.5), gridspec_kw=dict(width_ratios=[1, 1, 0.9]))
x1, y1, x2, y2 = gt_boxes[g].tolist(); pad = 0.5 * max(x2 - x1, y2 - y1)
for ax, bx, n, col, t in [(axes[0], Rc["props_all"][bm], n_before[g], "yellow", "BEFORE NMS"),
                          (axes[1], Rc["proposals"][am], n_after[g], "cyan", "AFTER NMS")]:
    imshow(ax, disp, f"GT{g} {t}: {n} boxes on this person")
    draw_boxes(ax, bx, color=col, lw=0.8, alpha=0.8)
    draw_boxes(ax, gt_boxes[g:g + 1], color="white", lw=2.5, ls="--")
    ax.set_xlim(max(0, x1 - pad), min(disp.shape[1], x2 + pad)); ax.set_ylim(min(disp.shape[0], y2 + pad), max(0, y1 - pad))
xs = np.arange(len(gt_boxes)); w = 0.38
axes[2].bar(xs - w / 2, n_before, w, color="gold", label="before NMS")
axes[2].bar(xs + w / 2, n_after, w, color="tab:cyan", label="after NMS")
axes[2].set_xticks(xs); axes[2].set_xticklabels([f"GT{i}" for i in xs]); axes[2].legend()
axes[2].set_ylabel(f"# boxes on each person (IoU>0.5, objectness>{SC_THR})"); axes[2].set_title("Duplicates per person: before vs after RPN NMS")
plt.tight_layout(); plt.show()

# proposal recall: GT 마다 top-K proposal 중 IoU 최대값
print("GT별 best IoU among top-K proposals")
for k in R:
    row = []
    for K in [10, 50, 100, 300, 1000]:
        best = box_iou(gt_boxes, R[k]["proposals"][:K]).max(1).values
        row.append(f"K={K}: {np.round(best.tolist(), 2)}")
    print(f"  [{k}]", " | ".join(row))

> 🤔 **생각해보기**
> 1. 학습 전(`init`)과 학습된(`coco`) 모델의 objectness heatmap 을 비교해 보세요. 학습된 모델은 어디가 밝게 빛나나요?
> 2. 확대 그림에서 NMS 전과 후, 한 사람에게 겹친 박스는 각각 몇 개인가요?
>
> 💡 RPN 의 NMS 는 IoU 0.7 기준이라 서로 덜 겹친 박스는 일부 남습니다 (FPN 레벨마다 따로 수행되기도 합니다). 남은 박스들은 2nd stage 뒤의 NMS (IoU 0.5) 에서 한 번 더 정리되어 사람당 박스 하나가 됩니다 (6절에서 확인).

## 6. 2nd stage: proposal 에 대한 classification + 추가 regression

2nd stage 는 RPN 이 넘긴 proposal 을 **새로운 anchor 처럼** 취급합니다. 정답 생성 규칙은 RPN 과 닮았지만 다릅니다.

| | RPN (1st stage) | RoI head (2nd stage) |
|---|---|---|
| 기준 박스 | 고정된 anchor | RPN 이 만든 proposal |
| positive 기준 | IoU ≥ 0.7 | IoU ≥ 0.5 |
| 샘플 수 / pos 비율 | 256 / 50% | 512 / 25% |
| 분류 대상 | object vs background | **C개 클래스 + background** |
| 회귀 | anchor → GT | proposal → GT (**클래스별** 오프셋) |
| 특이사항 | — | 학습 시 **GT 박스 자체를 proposal 에 추가** |

In [ ]:
roi = model_init.roi_heads
print(f"fg IoU ≥ {roi.proposal_matcher.high_threshold}, batch {roi.fg_bg_sampler.batch_size_per_image}, "
      f"pos fraction {roi.fg_bg_sampler.positive_fraction}")
print("RoI box_coder weights:", roi.box_coder.weights, "(RPN:", rpn.box_coder.weights, ")")

S = {}
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, k in zip(axes, R):
    seed_all()
    with torch.no_grad():
        sp, midx, slab, sreg = roi.select_training_samples([R[k]["proposals"].clone()], targets_t)
    sp, slab, sreg = sp[0], slab[0], sreg[0]
    S[k] = dict(props=sp, labels=slab, reg=sreg)
    p = slab > 0
    n_gtcopy = int((box_iou(sp[p], gt_boxes).max(1).values > 0.999).sum())
    print(f"[{k}] sampled {len(sp)} RoIs: positive {int(p.sum())} (그중 GT 복사본 {n_gtcopy}), negative {int((~p).sum())}")
    imshow(ax, disp, f"[{k}] 2nd-stage training RoIs\npositive {int(p.sum())} (GT copies {n_gtcopy}) / negative {int((~p).sum())}")
    draw_boxes(ax, sp[~p][torch.randperm(int((~p).sum()), device=sp.device)[:80]], color="red", lw=0.6, alpha=0.7)
    draw_boxes(ax, sp[p], color="lime", lw=1.2)
    draw_boxes(ax, gt_boxes, color="white", lw=2.5, ls="--")
plt.tight_layout(); plt.show()

k = "coco"; p = torch.where(S[k]["labels"] > 0)[0]
print("\n[coco] positive RoI 의 2nd-stage regression target 예시 (weights 10,10,5,5 가 곱해진 값):")
for i in p[:5].tolist():
    print(f"  RoI {i:3d}: class={CLASSES[int(S[k]['labels'][i])]:<10s} target={np.round(S[k]['reg'][i].tolist(), 3)}")

### RoIAlign: 크기가 제각각인 proposal → 고정 크기 7×7 feature
proposal 크기에 따라 FPN 레벨 $k = \lfloor 4 + \log_2(\sqrt{wh}/224) \rfloor$ (P2~P5 로 clamp) 에서 feature 를 잘라옵니다.

In [ ]:
pool = model_coco.roi_heads.box_roi_pool
sp, slab = S["coco"]["props"], S["coco"]["labels"]
pi2 = torch.where(slab > 0)[0][:4]; ni2 = torch.where(slab == 0)[0][:4]
sel = torch.cat([pi2, ni2])
with torch.no_grad():
    roi_feats = pool(feats_coco, [sp[sel]], image_list.image_sizes)          # (8, 256, 7, 7)
print("RoIAlign output:", tuple(roi_feats.shape))

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for j, i in enumerate(sel.tolist()):
    b = sp[i]; w, h = float(b[2] - b[0]), float(b[3] - b[1])
    lvl = int(np.clip(np.floor(4 + np.log2(np.sqrt(w * h) / 224 + 1e-6)), 2, 5))
    r0 = 0 if j < 4 else 2; c = j % 4
    x1, y1, x2, y2 = [int(v) for v in b.tolist()]
    imshow(axes[r0, c], disp[max(y1, 0):y2, max(x1, 0):x2],
           f"{'POS' if j < 4 else 'NEG'} RoI → P{lvl}\n{int(w)}x{int(h)} px")
    axes[r0 + 1, c].imshow(roi_feats[j].abs().mean(0).cpu(), cmap="viridis"); axes[r0 + 1, c].axis("off")
    axes[r0 + 1, c].set_title("7x7 RoIAlign feature", fontsize=9)
plt.tight_layout(); plt.show()

### Box head → classification + **클래스별** box refinement
학습된 COCO 모델로 proposal 이 어떻게 한 번 더 다듬어지는지 확인합니다. 점선 = proposal(1st stage), 실선 = 2nd stage 가 보정한 최종 박스.

In [ ]:
@torch.no_grad()
def roi_dissect(model, feats, proposals, image_sizes):
    rf = model.roi_heads.box_roi_pool(feats, [proposals], image_sizes)
    x = model.roi_heads.box_head(rf)                                  # (K, 1024)
    cls_logits, box_reg = model.roi_heads.box_predictor(x)            # (K, C), (K, 4C)
    refined = model.roi_heads.box_coder.decode(box_reg, [proposals])  # (K, C, 4)
    return cls_logits, box_reg, clip_boxes_to_image(refined, image_sizes[0])

props = R["coco"]["proposals"]
cls_logits, box_reg, refined = roi_dissect(model_coco, feats_coco, props, image_list.image_sizes)
probs = F.softmax(cls_logits, -1)
fg_score, fg_cls = probs[:, 1:].max(1); fg_cls += 1
top = fg_score.topk(12).indices

i_conf = top[0].item()                                   # 가장 확신하는 RoI
i_amb = (fg_score - 0.5).abs().argmin().item()           # 가장 애매한 RoI (fg score ≈ 0.5)

fig, axes = plt.subplots(1, 4, figsize=(26, 7), gridspec_kw=dict(width_ratios=[1, 1, 0.55, 0.55]))
imshow(axes[0], disp, "Top-12 RoIs: proposal (dashed) → refined box (solid)")
for n, i in enumerate(top.tolist()):
    c = plt.cm.tab20(n % 20)
    pb, rb = props[i], refined[i, fg_cls[i]]
    draw_boxes(axes[0], pb[None], color=[c], lw=1.2, ls="--")
    draw_boxes(axes[0], rb[None], color=[c], lw=2)
    axes[0].annotate("", xy=cxcywh(rb)[:2].tolist(), xytext=cxcywh(pb)[:2].tolist(),
                     arrowprops=dict(arrowstyle="->", color=c, lw=1.5))

# 2nd stage 후처리: score threshold + 클래스별 NMS
with torch.no_grad():
    db, ds_, dl = model_coco.roi_heads.postprocess_detections(cls_logits, box_reg, [props], image_list.image_sizes)
keep = ds_[0] > 0.5
imshow(axes[1], disp, f"After score>0.5 + per-class NMS: {int(keep.sum())} detections")
draw_boxes(axes[1], db[0][keep], color="lime", lw=2,
           texts=[f"{COCO_CLASSES[int(l)]} {s:.2f}" for l, s in zip(dl[0][keep], ds_[0][keep])])

draw_boxes(axes[1], props[i_amb][None], color="magenta", lw=2.5, ls=":")
for ax, i, tag, col in [(axes[2], i_conf, "most confident RoI", "tab:blue"), (axes[3], i_amb, "ambiguous RoI (magenta)", "magenta")]:
    p5 = probs[i].topk(5)
    ax.barh([COCO_CLASSES[int(c)] for c in p5.indices.tolist()][::-1], p5.values.cpu().numpy()[::-1], color=col)
    ax.set_xlim(0, 1); ax.set_title(f"{tag}\nsoftmax over {probs.shape[1]} classes (incl. bg)", fontsize=10)
plt.tight_layout(); plt.show()

> 🤔 **생각해보기** — 왼쪽 그림에서 한 사람에게 여러 개의 박스가 겹쳐 있나요? 가운데 그림(NMS 후)에서는 어떻게 바뀌었나요?

## 7. 학습 전 vs 완전히 학습된 모델 — 최종 inference 비교
흰 점선 = 데이터셋 GT. Penn-Fudan 은 **일부 보행자만 라벨링**되어 있어서, COCO 모델이 라벨 없는 사람을 찾아내는 경우도 보입니다.

In [ ]:
@torch.no_grad()
def detect(model, img): model.eval(); return model([img.to(device)])[0]

show_ids = [TRACE_IDX] + [i for i in range(len(val_ds)) if i != TRACE_IDX][:2]
fig, axes = plt.subplots(len(show_ids), 2, figsize=(14, 6 * len(show_ids)))
for r, i in enumerate(show_ids):
    im, t = val_ds[i]; d0, d1 = detect(model_init, im), detect(model_coco, im)
    imshow(axes[r, 0], im.permute(1, 2, 0), f"model_init: {len(d0['boxes'])} raw outputs (top-20 shown)")
    draw_boxes(axes[r, 0], d0["boxes"][:20], color="red", lw=1,
               texts=[f"{s:.2f}" for s in d0["scores"][:20]], fs=6)
    k1 = d1["scores"] > 0.5
    imshow(axes[r, 1], im.permute(1, 2, 0), f"model_coco: {int(k1.sum())} detections (score>0.5)")
    draw_boxes(axes[r, 1], d1["boxes"][k1], color="lime", lw=2,
               texts=[f"{COCO_CLASSES[int(l)]} {s:.2f}" for l, s in zip(d1["labels"][k1], d1["scores"][k1])])
    for ax in axes[r]: draw_boxes(ax, t["boxes"], color="white", lw=1.5, ls="--")
plt.tight_layout(); plt.show()

## 8. 직접 학습시키며 변화 관찰하기

`model_init` 을 fine-tune 하면서 `SNAP_EPOCHS` 시점의 가중치를 저장합니다. 학습이 끝나면 스냅샷마다
**① RPN objectness ② proposal ③ 최종 detection** 이 어떻게 바뀌는지 한눈에 비교합니다.

Faster R-CNN 의 loss 는 4개입니다: `loss_objectness`, `loss_rpn_box_reg` (1st stage) / `loss_classifier`, `loss_box_reg` (2nd stage).

In [ ]:
snapshots, history = {0: INIT_STATE}, []
model = build_init_model(); model.load_state_dict(INIT_STATE)
loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
                                     collate_fn=lambda b: tuple(zip(*b)))
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.SGD(params, lr=LR_FRCNN, momentum=0.9, weight_decay=1e-4)
total, warm = EPOCHS * len(loader), min(200, EPOCHS * len(loader) // 5)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda it: (it + 1) / warm if it < warm else 0.5 * (1 + math.cos(math.pi * (it - warm) / max(1, total - warm))))
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
t0 = time.time()
for ep in range(1, EPOCHS + 1):
    model.train()
    for imgs, tgts in loader:
        imgs = [i.to(device) for i in imgs]; tgts = [to_dev(t) for t in tgts]
        with torch.autocast(device_type=device.type, enabled=use_amp):
            ld = model(imgs, tgts)
            loss = sum(ld.values())
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 10.0)
        scaler.step(opt); scaler.update(); sched.step()
        history.append({k: v.item() for k, v in ld.items()})
    msg = " ".join(f"{k}={np.mean([h[k] for h in history[-len(loader):]]):.3f}" for k in history[-1])
    print(f"epoch {ep:2d}/{EPOCHS} ({time.time() - t0:5.0f}s)  {msg}")
    if ep in SNAP_EPOCHS:
        snapshots[ep] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
del model
if use_amp: torch.cuda.empty_cache()

In [ ]:
if history:
    keys = list(history[0].keys())
    fig, axes = plt.subplots(1, 4, figsize=(20, 3.5))
    for ax, k in zip(axes, keys):
        v = np.array([h[k] for h in history]); w = max(1, len(v) // 50)
        ax.plot(v, alpha=0.25, color="gray"); ax.plot(np.convolve(v, np.ones(w) / w, mode="valid"), color="tab:blue")
        ax.set_title(k + ("  (1st stage)" if "rpn" in k or "objectness" in k else "  (2nd stage)")); ax.set_xlabel("iteration")
    plt.tight_layout(); plt.show()

In [ ]:
model_snap = build_init_model().eval()
def load_snap(ep): model_snap.load_state_dict(snapshots[ep]); return model_snap.eval()

eps = sorted(snapshots)
fig, axes = plt.subplots(len(eps), 3, figsize=(18, 5.5 * len(eps)))
for r, ep in enumerate(eps):
    m = load_snap(ep); Rs = rpn_dissect(m, image_list)
    hm = torch.sigmoid(Rs["obj"][1][0]).max(0).values.cpu(); H, W = hm.shape; s = image_list.tensors.shape[-1] / W
    imshow(axes[r, 0], disp, f"epoch {ep}: RPN objectness (P3)")
    axes[r, 0].imshow(hm, cmap="jet", alpha=0.5, vmin=0, vmax=1, extent=(0, W * s, H * s, 0))
    axes[r, 0].set_xlim(0, disp.shape[1]); axes[r, 0].set_ylim(disp.shape[0], 0)
    imshow(axes[r, 1], disp, f"epoch {ep}: top-30 proposals"); draw_boxes(axes[r, 1], Rs["proposals"][:30], color="cyan", lw=1)
    d = detect(m, img); k1 = d["scores"] > 0.5
    imshow(axes[r, 2], img.permute(1, 2, 0), f"epoch {ep}: final detections (score>0.5): {int(k1.sum())}")
    draw_boxes(axes[r, 2], d["boxes"][k1], color="lime", lw=2,
               texts=[f"{CLASSES[int(l)]} {sc:.2f}" for l, sc in zip(d["labels"][k1], d["scores"][k1])])
    draw_boxes(axes[r, 2], tgt["boxes"], color="white", lw=1.5, ls="--")
plt.tight_layout(); plt.show()

### 정량 비교: proposal recall 과 detection precision / recall
- **Proposal recall@100** : GT 중 top-100 proposal 안에 IoU ≥ 0.5 (또는 0.7) 박스가 있는 비율 → **1st stage 의 품질**
- **Detection P / R** : score > 0.5 인 최종 검출을 IoU ≥ 0.5 로 GT 와 1:1 매칭 → **전체 파이프라인의 품질**

In [ ]:
@torch.no_grad()
def evaluate(model, ds, K=100, thr=0.5):
    n_gt = r50 = r70 = tp = fp = 0
    for i in range(len(ds)):
        im, t = ds[i]
        il, tt = model.transform([im.to(device)], [to_dev(t)])
        gt = tt[0]["boxes"]
        if len(gt) == 0: continue
        best = box_iou(gt, rpn_dissect(model, il)["proposals"][:K]).max(1).values
        r50 += int((best >= 0.5).sum()); r70 += int((best >= 0.7).sum()); n_gt += len(gt)
        d = detect(model, im); k = d["scores"] >= thr
        db, dl = d["boxes"][k], d["labels"][k]
        g0, gl = t["boxes"].to(device), t["labels"].to(device)
        used = torch.zeros(len(g0), dtype=torch.bool, device=device)
        ious = box_iou(db, g0) if len(db) else None
        for j in range(len(db)):                         # score 내림차순으로 greedy 매칭
            cand = (ious[j] >= 0.5) & (~used) & (gl == dl[j])
            if cand.any():
                used[torch.where(cand)[0][ious[j][cand].argmax()]] = True; tp += 1
            else: fp += 1
    return dict(prop_recall50=r50 / n_gt, prop_recall70=r70 / n_gt,
                det_precision=tp / max(1, tp + fp), det_recall=tp / n_gt)

rows = {ep: evaluate(load_snap(ep), val_ds) for ep in eps}
print(f"{'epoch':>6} | {'prop R@100 (0.5)':>16} | {'prop R@100 (0.7)':>16} | {'det P':>6} | {'det R':>6}")
for ep, m in rows.items():
    print(f"{ep:>6} | {m['prop_recall50']:>16.3f} | {m['prop_recall70']:>16.3f} | {m['det_precision']:>6.3f} | {m['det_recall']:>6.3f}")

In [ ]:
# 마지막으로 val 이미지 몇 장에서 epoch 0 vs 최종 epoch 비교
last = eps[-1]
fig, axes = plt.subplots(3, 2, figsize=(13, 17))
for r, i in enumerate(show_ids):
    im, t = val_ds[i]
    for c, ep in enumerate([0, last]):
        d = detect(load_snap(ep), im); k1 = d["scores"] > 0.5
        imshow(axes[r, c], im.permute(1, 2, 0), f"epoch {ep}: {int(k1.sum())} detections (score>0.5)")
        draw_boxes(axes[r, c], d["boxes"][k1], color="lime" if ep else "red", lw=2,
                   texts=[f"{CLASSES[int(l)]} {s:.2f}" for l, s in zip(d["labels"][k1], d["scores"][k1])])
        draw_boxes(axes[r, c], t["boxes"], color="white", lw=1.5, ls="--")
plt.tight_layout(); plt.show()

## 9. Part 1 정리

**핵심 요약**
- Anchor 는 학습 전부터 고정된 템플릿이고, **RPN 의 target(label, offset)은 anchor 와 GT 만으로 결정**됩니다.
- 학습되는 것은 "각 anchor 가 물체인지(objectness)"와 "얼마나 움직여야 하는지(offset)"를 **예측하는 능력**입니다.
- 2nd stage 는 proposal 을 새 기준 박스로 삼아 **같은 원리를 한 번 더** 적용하며, 이번엔 클래스까지 구분합니다.
- 이것이 "two-stage" 의 의미입니다: **어디에 있을까(1st) → 무엇이고 정확히 어디인가(2nd)**.

# Part 2. One-stage detector 해부하기 — RetinaNet

Part 1 에서 Faster R-CNN 은 **① RPN 이 후보를 고르고 → ② RoI head 가 후보마다 다시 분류·보정**하는 2단계였습니다.
One-stage detector 인 **RetinaNet** 은 이 둘을 합쳐, feature map 의 **모든 위치에서 클래스와 박스를 동시에** 예측합니다.
Part 1 과 같은 이미지·같은 방식으로 RetinaNet 을 해부하면서 **two-stage 와 무엇이 다른지**에 집중합니다.

## 10. 모델 준비와 구조 비교

In [ ]:
LEVEL_NAMES_R = ["P3", "P4", "P5", "P6", "P7"]      # RetinaNet 은 P3~P7 을 사용 (Faster R-CNN 은 P2~P6)
COCO_MAP = {"pedestrian": "person"}                 # 데이터셋 클래스 → COCO 학습 모델의 클래스

In [ ]:
def build_init():
    # ImageNet backbone + 랜덤 초기화 FPN/head (= 학습 시작 시점)
    seed_all()
    return retinanet_resnet50_fpn(weights=None, weights_backbone=ResNet50_Weights.IMAGENET1K_V1,
                                  num_classes=len(CLASSES), min_size=MIN_SIZE, max_size=MAX_SIZE).to(device).eval()

kw = dict(min_size=MIN_SIZE, max_size=MAX_SIZE)
retina_init = build_init()
retina_coco = retinanet_resnet50_fpn(weights=RetinaNet_ResNet50_FPN_Weights.COCO_V1, **kw).to(device).eval()
frcnn_coco  = model_coco                                    # Part 1 에서 불러온 COCO Faster R-CNN 재사용
COCO_CLASSES = RetinaNet_ResNet50_FPN_Weights.COCO_V1.meta["categories"]

FOCUS = CLASSES[1]                                   # "pedestrian"
COL = {"init": 1, "coco": COCO_CLASSES.index(COCO_MAP[FOCUS])}
print(f"focus class: {FOCUS}  (init 모델 column {COL['init']}, COCO 모델 column {COL['coco']} = '{COCO_MAP[FOCUS]}')")

def n_params(m): return sum(p.numel() for p in m.parameters()) / 1e6
for name, m in [("Faster R-CNN", frcnn_coco), ("RetinaNet", retina_coco)]:
    parts = ", ".join(f"{k}={n_params(v):.1f}M" for k, v in m.named_children() if n_params(v) > 0)
    print(f"{name:<13s}: {parts}")

위 출력에서 **Faster R-CNN 에만 `rpn`, `roi_heads`** 가 있고, RetinaNet 에는 그 자리에 `head` 하나만 있습니다.

In [ ]:
def draw_pipeline(ax, y, blocks, colors, title):
    x, w, h, gap = 0.0, 1.55, 0.9, 0.35
    for k, (b, c) in enumerate(zip(blocks, colors)):
        ax.add_patch(patches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05", facecolor=c, edgecolor="k"))
        ax.text(x + w / 2, y + h / 2, b, ha="center", va="center", fontsize=8.5)
        if k < len(blocks) - 1:
            ax.annotate("", xy=(x + w + gap, y + h / 2), xytext=(x + w + 0.05, y + h / 2),
                        arrowprops=dict(arrowstyle="->", lw=1.5))
        x += w + gap
    ax.text(-0.2, y + h / 2, title, ha="right", va="center", fontsize=11, fontweight="bold")

fig, ax = plt.subplots(figsize=(16, 4))
G, B, O, R_ = "#dddddd", "#cfe2ff", "#ffe0b3", "#ffd6d6"
draw_pipeline(ax, 1.4, ["Image", "Backbone\n+ FPN", "RPN head\nobjectness + offset\nper anchor",
                        "Top-K + NMS\n→ ~1000\nproposals", "RoIAlign\n7x7 per RoI", "RoI head\nclass + refine\nper RoI",
                        "NMS\n→ detections"], [G, B, O, O, R_, R_, G], "Two-stage\n(Faster R-CNN)")
draw_pipeline(ax, 0.0, ["Image", "Backbone\n+ FPN", "Dense head\nclass + box\nper anchor / point",
                        "score thr + NMS\n→ detections"], [G, B, O, G], "One-stage\n(RetinaNet)")
ax.set_xlim(-1.8, 13.5); ax.set_ylim(-0.3, 2.6); ax.axis("off"); plt.show()

## 11. Trace image 와 one-stage 해부 함수
torchvision RetinaNet 의 `forward` 를 단계별로 풀어 쓴 함수입니다.
```python
images, _   = model.transform(images)
features    = model.backbone(images.tensors)          # P3~P7
head_out    = model.head(features)                    # ← 여기서 class 와 box 를 "동시에" 예측
anchors     = model.anchor_generator(images, features)
detections  = model.postprocess_detections(head_out, anchors, images.image_sizes)   # score thr + NMS
```

In [ ]:
TRACE_IDX = next(i for i in range(len(val_ds)) if 2 <= len(val_ds[i][1]["boxes"]) <= 4)
img, tgt = val_ds[TRACE_IDX]
with torch.no_grad():
    image_list, targets_t = retina_init.transform([img.to(device)], [to_dev(tgt)])
gt_boxes = targets_t[0]["boxes"]; disp = to_display(image_list)

@torch.no_grad()
def os_dissect(model, image_list):
    model.eval()
    feats = list(model.backbone(image_list.tensors).values())
    head = {k: v[0] for k, v in model.head(feats).items()}        # batch 차원 제거: (HWA, ...)
    anchors = model.anchor_generator(image_list, feats)[0]
    grid = [tuple(f.shape[-2:]) for f in feats]
    A = head["cls_logits"].shape[0] // sum(h * w for h, w in grid)
    n_per = [h * w * A for h, w in grid]
    split = {k: list(v[None].split(n_per, dim=1)) for k, v in head.items()}
    dets = model.postprocess_detections(split, [list(anchors.split(n_per))], image_list.image_sizes)[0]
    scores = torch.sigmoid(head["cls_logits"])                       # (HWA, C)
    return dict(head=head, anchors=anchors, grid=grid, A=A, n_per=n_per, offsets=np.cumsum([0] + n_per),
                dets=dets, scores=scores)

def decode(model, R, idx):
    reg, anc = R["head"]["bbox_regression"][idx], R["anchors"][idx]
    boxes = model.box_coder.decode_single(reg, anc)
    return clip_boxes_to_image(boxes, image_list.image_sizes[0])

def level_map(R, s, l):
    # 레벨 l 의 (H, W) 맵 — 위치마다 A 개 anchor 중 최댓값
    H, W = R["grid"][l]; o = R["offsets"]
    return s[o[l]:o[l + 1]].view(H, W, R["A"]).max(-1).values.cpu()

def overlay(ax, hm, title, vmax=1.0, cmap="jet"):
    H, W = hm.shape; st = image_list.tensors.shape[-1] / W
    imshow(ax, disp, title)
    ax.imshow(hm, cmap=cmap, alpha=0.5, vmin=0, vmax=vmax, extent=(0, W * st, H * st, 0))
    ax.set_xlim(0, disp.shape[1]); ax.set_ylim(disp.shape[0], 0)

RR = {"init": os_dissect(retina_init, image_list), "coco": os_dissect(retina_coco, image_list)}
fig, ax = plt.subplots(figsize=(6, 6))
imshow(ax, disp, f"Trace image (val #{TRACE_IDX}) with GT")
draw_boxes(ax, gt_boxes, color=[GT_COLORS[i] for i in range(len(gt_boxes))], lw=2.5,
           texts=[f"GT{i}" for i in range(len(gt_boxes))]); plt.show()

## 12. Dense prediction — 모든 anchor 가 class 와 box 를 동시에 출력

RetinaNet 은 위치당 **9개 anchor** (3 scales × 3 ratios) 를 쓰고, anchor 마다
- `cls_logits` : **C 개 클래스 각각의 sigmoid 점수** (background 클래스 없음 — 모든 클래스가 0 에 가까우면 배경)
- `bbox_regression` : Part 1과 똑같은 (tx, ty, tw, th)

를 **한 번의 head forward 로** 냅니다. RPN 의 "objectness 1개" 가 "C 개 클래스 점수" 로 바뀐 것이라고 볼 수 있습니다.

In [ ]:
R = RR["coco"]
print("anchor sizes per level:", [tuple(round(float(s), 1) for s in sz) for sz in retina_coco.anchor_generator.sizes])
print("aspect ratios        :", retina_coco.anchor_generator.aspect_ratios[0])
for l, (H, W) in enumerate(R["grid"]):
    print(f"  {LEVEL_NAMES_R[l]}: {H:3d}x{W:3d} x A={R['A']} = {R['n_per'][l]:7,d} anchors")
print(f"\ncls_logits      : {tuple(R['head']['cls_logits'].shape)}  ← anchor 마다 {R['head']['cls_logits'].shape[1]} 클래스 점수")
print(f"bbox_regression : {tuple(R['head']['bbox_regression'].shape)}  ← anchor 마다 4개 offset")
print(f"→ head 한 번의 forward 로 {R['head']['cls_logits'].numel() + R['head']['bbox_regression'].numel():,} 개 숫자를 출력")

fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for r, k in enumerate(["init", "coco"]):
    s = RR[k]["scores"][:, COL[k]]
    for l in range(5):
        hm = level_map(RR[k], s, l)
        overlay(axes[r, l], hm, f"[{k}] {LEVEL_NAMES_R[l]} '{FOCUS}' score (max {hm.max():.3f})")
plt.tight_layout(); plt.show()
print("init 모델의 평균 점수:", float(RR["init"]["scores"].mean()), "→ Part 1의 RPN(0.5)과 달리 0.01 에서 시작합니다 (이유는 13절)")

In [ ]:
# 한 위치를 확대해서 보기: GT 와 IoU 가 가장 높은 anchor 가 있는 위치의 9개 anchor
R, m = RR["coco"], retina_coco
g = int((gt_boxes[:, 2:] - gt_boxes[:, :2]).prod(1).argmax())
best = int(box_iou(gt_boxes[g:g+1], R["anchors"])[0].argmax())
l = int(np.searchsorted(R["offsets"], best, side="right") - 1)
loc0 = R["offsets"][l] + ((best - R["offsets"][l]) // R["A"]) * R["A"]
idx = torch.arange(loc0, loc0 + R["A"], device=device)
sc = R["scores"][idx, COL["coco"]]
dec = decode(m, R, idx)

fig, axes = plt.subplots(1, 3, figsize=(20, 6.5), gridspec_kw=dict(width_ratios=[1, 1, 0.7]))
imshow(axes[0], disp, f"9 anchors at one {LEVEL_NAMES_R[l]} location (input templates)")
draw_boxes(axes[0], R["anchors"][idx], color=[plt.cm.tab10(i) for i in range(9)], lw=1.5, ls="--")
imshow(axes[1], disp, "same 9 anchors after regression (outputs)")
draw_boxes(axes[1], dec, color=[plt.cm.tab10(i) for i in range(9)], lw=1.5 ,
           texts=[f"{s:.2f}" for s in sc.tolist()])
for ax in axes[:2]: draw_boxes(ax, gt_boxes[g:g+1], color="white", lw=2.5)
axes[2].bar(range(9), sc.cpu(), color=[plt.cm.tab10(i) for i in range(9)])
axes[2].set_xlabel("anchor index at this location"); axes[2].set_ylim(0, 1)
axes[2].set_title(f"'{COCO_MAP[FOCUS]}' score of each anchor")
plt.tight_layout(); plt.show()

## 13. 학습 target 과 Focal Loss

**Label 할당 (RetinaNet)** : IoU ≥ 0.5 → positive, IoU < 0.4 → negative, 사이 → ignore. Part 1의 RPN 과 거의 같습니다.

**결정적인 차이 — 샘플링이 없습니다.** RPN 은 256개, RoI head 는 512개만 골라 loss 를 계산했지만,
one-stage 는 2단계가 없으므로 **모든 anchor (수만~수십만 개)** 가 classification loss 에 들어갑니다.
그런데 그중 positive 는 수십 개뿐입니다 → **극단적인 class imbalance**.

**Focal Loss** : $\mathrm{FL}(p_t) = -\alpha_t (1-p_t)^\gamma \log(p_t)$ — 이미 잘 맞히는(easy) 샘플의 loss 를 $(1-p_t)^\gamma$ 로 크게 줄입니다.

**Prior 초기화** : RetinaNet 은 학습 시작 시 모든 클래스 점수가 0.01 이 되도록 분류 head 를 초기화합니다. "처음에는 거의 다 배경" 이라고 가정하고 출발하는 것입니다 (12절에서 본 0.01 의 이유).

In [ ]:
R, m = RR["init"], retina_init
anchors = R["anchors"]
iou = box_iou(gt_boxes, anchors)
matched = m.proposal_matcher(iou)                 # ≥0: 매칭된 GT index, -1: negative, -2: ignore
pos, neg, ign = matched >= 0, matched == -1, matched == -2
print(f"fg IoU ≥ {m.proposal_matcher.high_threshold}, bg IoU < {m.proposal_matcher.low_threshold}")
print(f"positive {int(pos.sum()):,} | negative {int(neg.sum()):,} | ignore {int(ign.sum()):,} (총 {len(anchors):,})")
print(f"→ loss 에 쓰이는 anchor: {int((pos | neg).sum()):,} 개 (Part 1 RPN: 256개)  /  neg:pos ≈ {int(neg.sum()) / max(1, int(pos.sum())):,.0f} : 1")
print("  레벨별 positive:", {LEVEL_NAMES_R[l]: int(pos[R['offsets'][l]:R['offsets'][l+1]].sum()) for l in range(5)})

fig, ax = plt.subplots(figsize=(7, 7))
imshow(ax, disp, f"Positive anchors ({int(pos.sum())}), colored by matched GT")
pi = torch.where(pos)[0]
draw_boxes(ax, anchors[pi], color=[GT_COLORS[int(matched[i])] for i in pi], lw=1)
draw_boxes(ax, gt_boxes, color="white", lw=2.5, ls="--"); plt.show()

reg_t = m.box_coder.encode_single(gt_boxes[matched[pi]], anchors[pi])
print("regression target 예시 (Part 1 RPN 과 같은 방식):", np.round(reg_t[0].tolist(), 3))

In [ ]:
# (a) Focal loss 곡선
p = np.linspace(0.001, 1, 500)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for gma in [0, 0.5, 1, 2, 5]:
    axes[0].plot(p, -(1 - p) ** gma * np.log(p), label=f"gamma={gma}" + (" (= CE)" if gma == 0 else ""))
axes[0].set_xlabel("p_t (probability of the correct label)"); axes[0].set_ylabel("loss"); axes[0].set_ylim(0, 5)
axes[0].axvspan(0.6, 1, color="green", alpha=0.08); axes[0].text(0.62, 4.5, "well-classified (easy)", fontsize=9)
axes[0].legend(); axes[0].set_title("Focal loss vs cross entropy")

# (b) 이 이미지에서 실제로 계산해보기 — focus 클래스 column 하나에 대해
valid = pos | neg
y = (pos & (targets_t[0]["labels"][matched.clamp(min=0)] == 1)).float()
scen = {"uniform p=0.5\n(like RPN init)": torch.zeros_like(y),
        "prior p=0.01\n(RetinaNet init)": RR["init"]["head"]["cls_logits"][:, COL["init"]],
        "COCO-trained": RR["coco"]["head"]["cls_logits"][:, COL["coco"]]}
rows = []
for name, logit in scen.items():
    ce = F.binary_cross_entropy_with_logits(logit, y, reduction="none")[valid]
    fl = sigmoid_focal_loss(logit, y, alpha=0.25, gamma=2.0, reduction="none")[valid]
    yy = y[valid].bool()
    rows.append((name, ce[~yy].sum().item(), ce[yy].sum().item(), fl[~yy].sum().item(), fl[yy].sum().item()))
print(f"{'scenario':<28s} | {'CE neg':>10s} {'CE pos':>8s} {'neg share':>9s} | {'FL neg':>9s} {'FL pos':>8s} {'neg share':>9s}")
for n, cn, cp, fn, fp in rows:
    print(f"{n.replace(chr(10), ' '):<28s} | {cn:10.1f} {cp:8.1f} {cn / (cn + cp):9.1%} | {fn:9.2f} {fp:8.2f} {fn / (fn + fp):9.1%}")
x = np.arange(len(rows)); wd = 0.35
axes[1].bar(x - wd / 2, [r_[1] / (r_[1] + r_[2]) for r_ in rows], wd, label="CE", color="gray")
axes[1].bar(x + wd / 2, [r_[3] / (r_[3] + r_[4]) for r_ in rows], wd, label="Focal", color="tab:blue")
axes[1].set_xticks(x); axes[1].set_xticklabels([r_[0] for r_ in rows], fontsize=9)
axes[1].set_ylabel("share of total loss from negatives"); axes[1].set_ylim(0, 1); axes[1].legend()
axes[1].set_title("How much of the loss comes from background anchors?")
plt.tight_layout(); plt.show()

In [ ]:
# (c) 학습된 모델에서 anchor 별 loss 지도 — CE vs Focal (P3, 위치별 9개 anchor 합)
logit = scen["COCO-trained"]
ce_map = F.binary_cross_entropy_with_logits(logit, y, reduction="none") * valid
fl_map = sigmoid_focal_loss(logit, y, alpha=0.25, gamma=2.0, reduction="none") * valid
H, W = RR["coco"]["grid"][0]; n0 = RR["coco"]["n_per"][0]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (nm, lm) in zip(axes, [("Cross entropy", ce_map), ("Focal loss", fl_map)]):
    hm = lm[:n0].view(H, W, -1).sum(-1).cpu()
    overlay(ax, hm, f"{nm}: per-location loss on P3 (sum {float(lm.sum()):.1f})", vmax=float(ce_map[:n0].view(H, W, -1).sum(-1).max()), cmap="magma")
plt.tight_layout(); plt.show()

> 🤔 **생각해보기**
> 1. 이 이미지에서 positive anchor 와 negative anchor 는 각각 몇 개였나요? Part 1의 RPN 은 그중 몇 개만 골라 loss 에 썼나요?
> 2. 위 막대그래프에서 일반 cross entropy(회색)와 focal loss(파란색)를 비교해 보세요. 배경 anchor 가 차지하는 loss 비중이 어떻게 달라졌나요?

## 14. 후처리: dense 출력 → 최종 검출
2-stage 는 RPN 에서 한 번, RoI head 뒤에서 한 번 걸러냈지만, one-stage 는 **score threshold + NMS 한 번** 으로 끝납니다.

In [ ]:
R, m = RR["coco"], retina_coco
s = R["scores"][:, COL["coco"]]
n_thr = int((s > m.score_thresh).sum())
top = s.topk(300).indices
cand = decode(m, R, top)
d = R["dets"]; keep = d["scores"] > 0.5
print(f"RetinaNet: {len(s):,} anchors → score>{m.score_thresh}: {n_thr:,} → NMS({m.nms_thresh}) 후 최종 {len(d['boxes'])} 개 (score>0.5: {int(keep.sum())})")

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
imshow(axes[0], disp, "Top-300 dense predictions BEFORE NMS")
draw_boxes(axes[0], cand, color="yellow", lw=0.6, alpha=0.6)
imshow(axes[1], disp, f"After NMS, score>0.5: {int(keep.sum())} detections")
draw_boxes(axes[1], d["boxes"][keep], color="lime", lw=2,
           texts=[f"{COCO_CLASSES[int(c)]} {v:.2f}" for c, v in zip(d["labels"][keep], d["scores"][keep])])
for ax in axes: draw_boxes(ax, gt_boxes, color="white", lw=1.5, ls="--")
plt.tight_layout(); plt.show()

| | Faster R-CNN (Part 1) | RetinaNet |
|---|---|---|
| 1차 출력 | 약 12만 anchor × (objectness 1 + offset 4) | 약 12만 anchor × (**C 클래스** + offset 4) |
| 중간 단계 | top-K + NMS → 1000 proposals → RoIAlign → RoI head | **없음** |
| 클래스 결정 | 2nd stage (softmax, bg 포함) | head 에서 바로 (sigmoid, bg 없음) |
| box 보정 횟수 | 2번 (anchor→proposal→box) | 1번 (anchor→box) |
| 학습 시 샘플링 | 256 / 512 개 | 없음 → focal loss |

## 15. 학습 전 vs 완전히 학습된 모델 — 최종 inference 비교

In [ ]:
@torch.no_grad()
def detect(model, im): model.eval(); return model([im.to(device)])[0]

def show_det(ax, im, d, title, names, thr=0.5, color="lime"):
    k = d["scores"] > thr
    imshow(ax, im.permute(1, 2, 0), f"{title}: {int(k.sum())} (score>{thr})")
    draw_boxes(ax, d["boxes"][k], color=color, lw=2,
               texts=[f"{names[int(c)]} {v:.2f}" for c, v in zip(d["labels"][k], d["scores"][k])])

show_ids = [TRACE_IDX] + [i for i in range(len(val_ds)) if i != TRACE_IDX][:2]
cols = [("RetinaNet init", retina_init, CLASSES, 0.05), ("RetinaNet COCO", retina_coco, COCO_CLASSES, 0.5),
        ("Faster R-CNN COCO (two-stage)", frcnn_coco, COCO_CLASSES, 0.5)]
fig, axes = plt.subplots(len(show_ids), 3, figsize=(19, 5.5 * len(show_ids)))
for r, i in enumerate(show_ids):
    im, t = val_ds[i]
    for c, (nm, m, names, thr) in enumerate(cols):
        show_det(axes[r, c], im, detect(m, im), nm, names, thr, "red" if "init" in nm else "lime")
        draw_boxes(axes[r, c], t["boxes"], color="white", lw=1.5, ls="--")
plt.tight_layout(); plt.show()

> Part 1의 Faster R-CNN 학습 전 모델은 점수 0.5 근처의 박스를 수백 개 쏟아냈지만, RetinaNet 학습 전 모델은 threshold 0.05 로 낮춰도 **거의 아무것도 내지 않습니다.** 13절의 prior 초기화 때문입니다.

## 16. 직접 학습시키며 변화 관찰하기
RetinaNet 을 ImageNet backbone 부터 직접 학습합니다. 손실은 `classification`(focal loss) + `bbox_regression` **두 가지** 입니다
(Part 1의 Faster R-CNN 은 1st stage 2개 + 2nd stage 2개 = 4가지였습니다).

In [ ]:
INIT_STATE_R = {k: v.detach().cpu().clone() for k, v in build_init().state_dict().items()}
snapshots_r, history_r = {0: INIT_STATE_R}, []
model = build_init(); model.load_state_dict(INIT_STATE_R)
loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
                                     collate_fn=lambda b: tuple(zip(*b)))
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.SGD(params, lr=LR_RETINA, momentum=0.9, weight_decay=1e-4)
total, warm = EPOCHS * len(loader), min(200, EPOCHS * len(loader) // 5)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda it: (it + 1) / warm if it < warm else 0.5 * (1 + math.cos(math.pi * (it - warm) / max(1, total - warm))))
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
t0 = time.time()
for ep in range(1, EPOCHS + 1):
    model.train()
    for imgs, tgts in loader:
        imgs = [i.to(device) for i in imgs]; tgts = [to_dev(t) for t in tgts]
        with torch.autocast(device_type=device.type, enabled=use_amp):
            ld = model(imgs, tgts); loss = sum(ld.values())
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 10.0)
        scaler.step(opt); scaler.update(); sched.step()
        history_r.append({k: v.item() for k, v in ld.items()})
    msg = " ".join(f"{k}={np.mean([h[k] for h in history_r[-len(loader):]]):.3f}" for k in history_r[-1])
    print(f"epoch {ep:2d}/{EPOCHS} ({time.time() - t0:5.0f}s)  {msg}")
    if ep in SNAP_EPOCHS:
        snapshots_r[ep] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
del model
if use_amp: torch.cuda.empty_cache()

In [ ]:
if history_r:
    keys = list(history_r[0].keys())
    fig, axes = plt.subplots(1, len(keys), figsize=(6 * len(keys), 3.5))
    for ax, k in zip(np.atleast_1d(axes), keys):
        v = np.array([h[k] for h in history_r]); w = max(1, len(v) // 50)
        ax.plot(v, alpha=0.25, color="gray"); ax.plot(np.convolve(v, np.ones(w) / w, mode="valid"), color="tab:blue")
        ax.set_title(k); ax.set_xlabel("iteration")
    plt.tight_layout(); plt.show()

In [ ]:
model_snap_r = build_init()
def load_snap_r(ep): model_snap_r.load_state_dict(snapshots_r[ep]); return model_snap_r.eval()

eps_r = sorted(snapshots_r)
fig, axes = plt.subplots(len(eps_r), 3, figsize=(18, 5.5 * len(eps_r)))
for r, ep in enumerate(eps_r):
    m = load_snap_r(ep); Rs = os_dissect(m, image_list); s = Rs["scores"][:, 1]
    overlay(axes[r, 0], level_map(Rs, s, 0), f"epoch {ep}: dense '{FOCUS}' score (P3, max {float(s.max()):.2f})")
    imshow(axes[r, 1], disp, f"epoch {ep}: top-100 predictions before NMS")
    draw_boxes(axes[r, 1], decode(m, Rs, s.topk(100).indices), color="yellow", lw=0.6)
    show_det(axes[r, 2], img, detect(m, img), f"epoch {ep}: final", CLASSES)
    draw_boxes(axes[r, 2], tgt["boxes"], color="white", lw=1.5, ls="--")
plt.tight_layout(); plt.show()

In [ ]:
@torch.no_grad()
def evaluate_r(model, ds, thr=0.5):
    n_gt = tp = fp = 0
    for i in range(len(ds)):
        im, t = ds[i]
        d = detect(model, im); k = d["scores"] >= thr
        db, dl = d["boxes"][k], d["labels"][k]
        g0, gl = t["boxes"].to(device), t["labels"].to(device); n_gt += len(g0)
        used = torch.zeros(len(g0), dtype=torch.bool, device=device)
        ious = box_iou(db, g0) if len(db) and len(g0) else None
        for j in range(len(db)):
            cand = (ious[j] >= 0.5) & (~used) & (gl == dl[j]) if ious is not None else torch.zeros(0, dtype=torch.bool)
            if cand.any(): used[torch.where(cand)[0][ious[j][cand].argmax()]] = True; tp += 1
            else: fp += 1
    return tp / max(1, tp + fp), tp / max(1, n_gt)

print("epoch | det P | det R   (score>0.5, IoU≥0.5)")
for ep in eps_r:
    P, Rc = evaluate_r(load_snap_r(ep), val_ds)
    print(f"{ep:5d} | {P:.3f} | {Rc:.3f}")

## 17. 전체 정리 — two-stage vs one-stage

**핵심 요약**
- One-stage 는 RPN 의 "objectness" 자리에 **클래스 점수**를 바로 두어, proposal → RoIAlign → RoI head 단계를 없앴습니다.
- 그 대신 **모든 anchor 가 loss 에 참여**하게 되어 배경이 압도적으로 많아지고, **focal loss 와 prior 초기화**로 이를 해결합니다.
- 박스 보정은 한 번 (anchor → box), 후처리는 score threshold + NMS 한 번으로 끝납니다.